### 1. Import Libraries 

#### Name:- Arpita Anap
Roll No:- 05
Div:- A

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping

### 2. Load Dataset

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Convert TotalCharges to Numeric

In [3]:
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

df['TotalCharges'].dtype

dtype('float64')

In [4]:
# Remove missing values 
df = df.dropna()

print("Dataset shape after removing missing values:", df.shape)

Dataset shape after removing missing values: (7032, 21)


In [6]:
# ==========================================
# CELL 7: TARGET VARIABLE ANALYSIS
# ==========================================

# Display number of customers in each class
print("Churn counts:")
print(df['Churn'].value_counts())

# Display percentage of each class
print("\nChurn percentage:")
print(df['Churn'].value_counts(normalize=True) * 100)

Churn counts:
Churn
No     5163
Yes    1869
Name: count, dtype: int64

Churn percentage:
Churn
No     73.421502
Yes    26.578498
Name: proportion, dtype: float64


In [7]:
# ==========================================
# CELL 9: REMOVE ID AND ENCODE TARGET
# ==========================================

# Remove customerID because it is only an identifier
df = df.drop('customerID', axis=1)

# Convert target variable into binary numerical values:
# No  -> 0
# Yes -> 1

df['Churn'] = df['Churn'].map({
    'No': 0,
    'Yes': 1
})

# Check the result
print("Target values after encoding:")
print(df['Churn'].value_counts())

# Display first few rows
display(df.head())

Target values after encoding:
Churn
0    5163
1    1869
Name: count, dtype: int64


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [8]:
# ==========================================
# CELL 11: FEATURE / TARGET SPLIT
# ==========================================

# X contains all input features
X = df.drop('Churn', axis=1)

# y contains the target variable
y = df['Churn']

print("Original feature shape:", X.shape)
print("Target shape:", y.shape)


# ==========================================
# ONE-HOT ENCODING
# ==========================================

# Convert categorical columns into numerical columns.
#
# Example:
# Contract:
# Month-to-month
# One year
# Two year
#
# becomes numerical binary columns.

X = pd.get_dummies(
    X,
    drop_first=True
)

# Convert all features to float.
# This ensures compatibility with TensorFlow.
X = X.astype(float)

print("\nFeature shape after encoding:", X.shape)

# Display the processed features
display(X.head())

Original feature shape: (7032, 19)
Target shape: (7032,)

Feature shape after encoding: (7032, 30)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0.0,1.0,29.85,29.85,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.0,34.0,56.95,1889.50,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,0.0,2.0,53.85,108.15,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,0.0,45.0,42.30,1840.75,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.0,2.0,70.70,151.65,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [9]:
# ==========================================
# CELL 13: TRAIN-TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,       # 20% testing data
    stratify=y,           # Preserve class distribution
    random_state=42       # Reproducible result
)

print("Training data shape:", X_train.shape)
print("Testing data shape :", X_test.shape)


# ==========================================
# FEATURE SCALING
# ==========================================

# StandardScaler standardizes features approximately
# around mean = 0 and standard deviation = 1.

scaler = StandardScaler()

# IMPORTANT:
# fit_transform() is used ONLY on training data.
# The scaler learns mean and standard deviation from X_train.

X_train = scaler.fit_transform(X_train)

# For test data, only transform() is used.
# We use the same scaling parameters learned from training data.

X_test = scaler.transform(X_test)

print("\nFeature scaling completed.")

# Number of input features for the ANN
input_features = X_train.shape[1]

print("Number of input features:", input_features)

Training data shape: (5625, 30)
Testing data shape : (1407, 30)

Feature scaling completed.
Number of input features: 30


In [10]:
# ==========================================
# CELL 15: BASIC ANN MODEL
# ==========================================

basic_model = Sequential([

    # First hidden layer
    # 64 neurons with ReLU activation
    Dense(
        64,
        activation='relu',
        input_shape=(input_features,)
    ),

    # Second hidden layer
    Dense(
        32,
        activation='relu'
    ),

    # Output layer
    # One neuron because this is binary classification
    # Sigmoid gives probability between 0 and 1
    Dense(
        1,
        activation='sigmoid'
    )
])

# Display model architecture
basic_model.summary()

C:\Users\arpit\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,097 (16.00 KB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# ==========================================
# CELL 17: COMPILE BASIC ANN
# ==========================================

basic_model.compile(

    # Adam optimization algorithm
    optimizer=Adam(learning_rate=0.001),

    # Loss function for binary classification
    loss='binary_crossentropy',

    # Evaluation metric
    metrics=['accuracy']
)

In [12]:
# ==========================================
# CELL 19: TRAIN BASIC ANN
# ==========================================

basic_history = basic_model.fit(

    # Training input
    X_train,

    # Training target
    y_train,

    # Maximum training epochs
    epochs=50,

    # Number of samples processed at a time
    batch_size=32,

    # 20% of training data is used for validation
    validation_split=0.2,

    # Display training progress
    verbose=1
)

Epoch 1/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7636 - loss: 0.4677 - val_accuracy: 0.7831 - val_loss: 0.4158
Epoch 2/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8011 - loss: 0.4238 - val_accuracy: 0.7956 - val_loss: 0.4106
Epoch 3/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8064 - loss: 0.4130 - val_accuracy: 0.7982 - val_loss: 0.4119
Epoch 4/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8129 - loss: 0.4078 - val_accuracy: 0.8027 - val_loss: 0.4125
Epoch 5/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8151 - loss: 0.4015 - val_accuracy: 0.8027 - val_loss: 0.4203
Epoch 6/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8184 - loss: 0.3988 - val_accuracy: 0.7902 - val_loss: 0.4180
Epoch 7/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8140 - loss: 0.3944 - val_accuracy: 0.7938 - val_loss: 0.4181
Epoch 8/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8229 - loss: 0.3895 - val_accuracy: 0.

In [13]:
# ==========================================
# CELL 21: BASIC ANN EVALUATION
# ==========================================

# Generate probability predictions
y_prob_basic = basic_model.predict(X_test)

# Convert probability into class:
# > 0.5 → 1 (Churn)
# <= 0.5 → 0 (No Churn)

y_pred_basic = (
    y_prob_basic > 0.5
).astype(int).ravel()


# ==========================================
# CALCULATE METRICS
# ==========================================

basic_accuracy = accuracy_score(
    y_test,
    y_pred_basic
)

basic_precision = precision_score(
    y_test,
    y_pred_basic
)

basic_recall = recall_score(
    y_test,
    y_pred_basic
)

basic_f1 = f1_score(
    y_test,
    y_pred_basic
)


# ==========================================
# DISPLAY RESULTS
# ==========================================

print("===== BASIC ANN RESULTS =====")

print("Accuracy :", basic_accuracy)
print("Precision:", basic_precision)
print("Recall   :", basic_recall)
print("F1-Score :", basic_f1)


# ==========================================
# CONFUSION MATRIX
# ==========================================

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_basic))


# Detailed classification report
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_basic
    )
)

44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
===== BASIC ANN RESULTS =====
Accuracy : 0.7583511016346838
Precision: 0.5537974683544303
Recall   : 0.4679144385026738
F1-Score : 0.5072463768115942

Confusion Matrix:
[[892 141]
 [199 175]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.86      0.84      1033
           1       0.55      0.47      0.51       374

    accuracy                           0.76      1407
   macro avg       0.69      0.67      0.67      1407
weighted avg       0.75      0.76      0.75      1407



In [14]:
# ==========================================
# CELL 23: REGULARIZED ANN
# ==========================================

regularized_model = Sequential([

    # First Dense layer
    Dense(
        64,
        activation='relu',
        input_shape=(input_features,)
    ),

    # Batch Normalization
    # Helps stabilize the activations
    BatchNormalization(),

    # Dropout
    # Randomly disables 30% neurons during training
    Dropout(0.3),


    # Second Dense layer
    Dense(
        32,
        activation='relu'
    ),

    # Batch Normalization
    BatchNormalization(),

    # Dropout
    Dropout(0.3),


    # Output layer
    Dense(
        1,
        activation='sigmoid'
    )
])

# Display architecture
regularized_model.summary()

C:\Users\arpit\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,481 (17.50 KB)

 Trainable params: 4,289 (16.75 KB)

 Non-trainable params: 192 (768.00 B)

In [15]:
# ==========================================
# CELL 25: COMPILE ADAM MODEL
# ==========================================

regularized_model.compile(

    # Adam optimizer
    optimizer=Adam(
        learning_rate=0.001
    ),

    # Binary classification loss
    loss='binary_crossentropy',

    # Accuracy metric
    metrics=['accuracy']
)

In [16]:
# ==========================================
# CELL 27: EARLY STOPPING
# ==========================================

early_stop = EarlyStopping(

    # Monitor validation loss
    monitor='val_loss',

    # Stop after 5 epochs without improvement
    patience=5,

    # Restore the best model weights
    restore_best_weights=True
)

In [17]:
# ==========================================
# CELL 29: TRAIN ADAM MODEL
# ==========================================

adam_history = regularized_model.fit(

    X_train,
    y_train,

    # Maximum number of epochs
    epochs=100,

    # Process 32 samples at a time
    batch_size=32,

    # Use 20% training data for validation
    validation_split=0.2,

    # Stop training if validation loss stops improving
    callbacks=[early_stop],

    verbose=1
)

Epoch 1/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6920 - loss: 0.6493 - val_accuracy: 0.8009 - val_loss: 0.4515
Epoch 2/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7587 - loss: 0.5176 - val_accuracy: 0.7911 - val_loss: 0.4235
Epoch 3/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7749 - loss: 0.4770 - val_accuracy: 0.7911 - val_loss: 0.4211
Epoch 4/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7762 - loss: 0.4667 - val_accuracy: 0.7964 - val_loss: 0.4135
Epoch 5/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7858 - loss: 0.4524 - val_accuracy: 0.7982 - val_loss: 0.4126
Epoch 6/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7880 - loss: 0.4520 - val_accuracy: 0.7947 - val_loss: 0.4155
Epoch 7/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7916 - loss: 0.4366 - val_accuracy: 0.7947 - val_loss: 0.4111
Epoch 8/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7880 - loss: 0.4412 - val_accu

In [18]:
# ==========================================
# CELL 31: EVALUATE ADAM MODEL
# ==========================================

# Generate probabilities
y_prob_adam = regularized_model.predict(X_test)

# Convert probabilities to classes
y_pred_adam = (
    y_prob_adam > 0.5
).astype(int).ravel()


# Calculate evaluation metrics
adam_accuracy = accuracy_score(
    y_test,
    y_pred_adam
)

adam_precision = precision_score(
    y_test,
    y_pred_adam
)

adam_recall = recall_score(
    y_test,
    y_pred_adam
)

adam_f1 = f1_score(
    y_test,
    y_pred_adam
)


# Display results
print("===== REGULARIZED ANN - ADAM =====")

print("Accuracy :", adam_accuracy)
print("Precision:", adam_precision)
print("Recall   :", adam_recall)
print("F1-Score :", adam_f1)


# Confusion Matrix
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_adam
    )
)


# Classification report
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_adam
    )
)

44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
===== REGULARIZED ANN - ADAM =====
Accuracy : 0.798862828713575
Precision: 0.6630824372759857
Recall   : 0.4946524064171123
F1-Score : 0.5666156202143952

Confusion Matrix:
[[939  94]
 [189 185]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1033
           1       0.66      0.49      0.57       374

    accuracy                           0.80      1407
   macro avg       0.75      0.70      0.72      1407
weighted avg       0.79      0.80      0.79      1407



In [19]:
# ==========================================
# CELL 33: SGD REGULARIZED ANN
# ==========================================

sgd_model = Sequential([

    # First Dense layer
    Dense(
        64,
        activation='relu',
        input_shape=(input_features,)
    ),

    # Batch Normalization
    BatchNormalization(),

    # Dropout regularization
    Dropout(0.3),


    # Second Dense layer
    Dense(
        32,
        activation='relu'
    ),

    # Batch Normalization
    BatchNormalization(),

    # Dropout regularization
    Dropout(0.3),


    # Binary classification output
    Dense(
        1,
        activation='sigmoid'
    )
])

# Display architecture
sgd_model.summary()

C:\Users\arpit\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,481 (17.50 KB)

 Trainable params: 4,289 (16.75 KB)

 Non-trainable params: 192 (768.00 B)

In [20]:
# ==========================================
# CELL 35: COMPILE SGD MODEL
# ==========================================

sgd_model.compile(

    # Stochastic Gradient Descent optimizer
    optimizer=SGD(
        learning_rate=0.01
    ),

    # Binary classification loss
    loss='binary_crossentropy',

    # Accuracy metric
    metrics=['accuracy']
)

In [21]:
# ==========================================
# CELL 37: TRAIN SGD MODEL
# ==========================================

sgd_history = sgd_model.fit(

    X_train,
    y_train,

    # Maximum epochs
    epochs=100,

    # Batch size
    batch_size=32,

    # Validation data
    validation_split=0.2,

    # Early stopping
    callbacks=[early_stop],

    verbose=1
)

Epoch 1/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.6360 - loss: 0.7141 - val_accuracy: 0.7840 - val_loss: 0.4800
Epoch 2/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7333 - loss: 0.5557 - val_accuracy: 0.7947 - val_loss: 0.4448
Epoch 3/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7507 - loss: 0.5198 - val_accuracy: 0.7982 - val_loss: 0.4374
Epoch 4/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7611 - loss: 0.4956 - val_accuracy: 0.8027 - val_loss: 0.4312
Epoch 5/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7693 - loss: 0.4761 - val_accuracy: 0.7991 - val_loss: 0.4259


In [22]:
# ==========================================
# CELL 39: EVALUATE SGD MODEL
# ==========================================

# Generate probability predictions
y_prob_sgd = sgd_model.predict(X_test)

# Convert probabilities to binary classes
y_pred_sgd = (
    y_prob_sgd > 0.5
).astype(int).ravel()


# Calculate metrics
sgd_accuracy = accuracy_score(
    y_test,
    y_pred_sgd
)

sgd_precision = precision_score(
    y_test,
    y_pred_sgd
)

sgd_recall = recall_score(
    y_test,
    y_pred_sgd
)

sgd_f1 = f1_score(
    y_test,
    y_pred_sgd
)


# Display results
print("===== REGULARIZED ANN - SGD =====")

print("Accuracy :", sgd_accuracy)
print("Precision:", sgd_precision)
print("Recall   :", sgd_recall)
print("F1-Score :", sgd_f1)


# Confusion Matrix
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred_sgd
    )
)


# Classification Report
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_sgd
    )
)

44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
===== REGULARIZED ANN - SGD =====
Accuracy : 0.759772565742715
Precision: 0.5548780487804879
Recall   : 0.48663101604278075
F1-Score : 0.5185185185185185

Confusion Matrix:
[[887 146]
 [192 182]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.86      0.84      1033
           1       0.55      0.49      0.52       374

    accuracy                           0.76      1407
   macro avg       0.69      0.67      0.68      1407
weighted avg       0.75      0.76      0.75      1407



In [23]:
# ==========================================
# CELL 41: MODEL COMPARISON
# ==========================================

comparison = pd.DataFrame({

    'Model': [
        'Basic ANN - Adam',
        'Regularized ANN - Adam',
        'Regularized ANN - SGD'
    ],

    'Accuracy': [
        basic_accuracy,
        adam_accuracy,
        sgd_accuracy
    ],

    'Precision': [
        basic_precision,
        adam_precision,
        sgd_precision
    ],

    'Recall': [
        basic_recall,
        adam_recall,
        sgd_recall
    ],

    'F1-Score': [
        basic_f1,
        adam_f1,
        sgd_f1
    ]
})


# Display comparison
display(comparison)

,Model,Accuracy,Precision,Recall,F1-Score
0,Basic ANN - Adam,0.758351,0.553797,0.467914,0.507246
1,Regularized ANN - Adam,0.798863,0.663082,0.494652,0.566616
2,Regularized ANN - SGD,0.759773,0.554878,0.486631,0.518519


In [24]:
# ==========================================
# CELL 43: CLASSIFICATION REPORTS
# ==========================================

print("=" * 50)
print("BASIC ANN - ADAM")
print("=" * 50)

print(
    classification_report(
        y_test,
        y_pred_basic
    )
)


print("=" * 50)
print("REGULARIZED ANN - ADAM")
print("=" * 50)

print(
    classification_report(
        y_test,
        y_pred_adam
    )
)


print("=" * 50)
print("REGULARIZED ANN - SGD")
print("=" * 50)

print(
    classification_report(
        y_test,
        y_pred_sgd
    )
)

BASIC ANN - ADAM
              precision    recall  f1-score   support

           0       0.82      0.86      0.84      1033
           1       0.55      0.47      0.51       374

    accuracy                           0.76      1407
   macro avg       0.69      0.67      0.67      1407
weighted avg       0.75      0.76      0.75      1407

REGULARIZED ANN - ADAM
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1033
           1       0.66      0.49      0.57       374

    accuracy                           0.80      1407
   macro avg       0.75      0.70      0.72      1407
weighted avg       0.79      0.80      0.79      1407

REGULARIZED ANN - SGD
              precision    recall  f1-score   support

           0       0.82      0.86      0.84      1033
           1       0.55      0.49      0.52       374

    accuracy                           0.76      1407
   macro avg       0.69      0.67      0.68      1407
weighted avg